# Basic Test

In [ ]:
import torch

from llm_circuits.circuits.replacement_model import compare_models
from llm_circuits.instrumentation.chat import prepare_messages
from llm_circuits.models.qwen3 import load_qwen3
from llm_circuits.transcoders.circuit_tracer_loader import load_transcoder

model, tokenizer = load_qwen3("0.6b")
model.eval()
print(f"Model: {type(model).__name__}, device: {model.device}, dtype: {model.dtype}")

loaded = load_transcoder("0.6b", device=model.device, dtype=model.dtype, lazy_decoder=True)
tc = loaded.transcoder
print(f"Transcoder: {type(loaded.transcoder).__name__}")
print(f"Repo: {loaded.repo_id}")
print(f"Config keys: {list(loaded.config.keys())}")

In [ ]:
prompt = 'Answer immediately: What is the capital city of China?'

# --- Tokenize via chat template -------------------------------------------
messages, n_bos_tokens, template_kwargs = prepare_messages(prompt, "qwen3")
input_ids = tokenizer.apply_chat_template(
    messages, return_tensors="pt", add_generation_prompt=True, **template_kwargs,
).to(model.device)
tokens = [tokenizer.decode(t) for t in input_ids[0]]

print(f"\nPrompt: {prompt!r}")
print(f"Tokens: {tokens}\n")

In [ ]:
with torch.no_grad():
    result = compare_models(
        model, tc, input_ids, n_bos_tokens=n_bos_tokens,
    )

In [ ]:
# --- Print per-position metrics -------------------------------------------
orig_preds = result.original_logits.argmax(dim=-1)
repl_preds = result.replacement_logits.argmax(dim=-1)

print(
    f"{'Pos':>3}  {'Token':>12}  {'Orig pred':>12}  {'Repl pred':>12}"
    f"  {'KL div':>10}  {'Cos sim':>10}  {'Top-1':>6}"
)
print("-" * 78)
for i, tok in enumerate(tokens):
    orig_tok = tokenizer.decode(orig_preds[i].item())
    repl_tok = tokenizer.decode(repl_preds[i].item())
    kl = result.kl_divergence[i].item()
    cos = result.cosine_similarity[i].item()
    agree = "yes" if result.top1_agreement[i].item() else "NO"
    print(
        f"{i:3d}  {tok:>12s}  {orig_tok:>12s}  {repl_tok:>12s}"
        f"  {kl:10.4f}  {cos:10.4f}  {agree:>6s}"
    )

# --- Summary --------------------------------------------------------------
mean_kl = result.kl_divergence.mean().item()
mean_cos = result.cosine_similarity.mean().item()
pct_agree = result.top1_agreement.float().mean().item() * 100

print(f"\nMean KL divergence:   {mean_kl:.4f}")
print(f"Mean cosine sim:      {mean_cos:.4f}")
print(f"Top-1 agreement:      {pct_agree:.1f}%")

# --- Per-layer reconstruction error ---------------------------------------
if result.reconstruction_errors:
    print(f"\n{'Layer':>5}  {'Mean L2 error':>14}")
    print("-" * 22)
    for layer_idx in sorted(result.reconstruction_errors):
        err = result.reconstruction_errors[layer_idx].mean().item()
        print(f"{layer_idx:5d}  {err:14.4f}")

In [ ]:
output_ids_org = torch.argmax(result.original_logits, axis=1)
text = tokenizer.decode(output_ids_org)
print(output_ids_org, text)

output_ids_rc = torch.argmax(result.replacement_logits, axis=1)
text_rc = tokenizer.decode(output_ids_rc)
print(output_ids_rc, text_rc)

In [ ]:
dir(result)
print(result.original_activations[20])
print(result.replacement_activations[20])

In [ ]:
dir(loaded)